# Scratch — 2026 mid-year refresh

**One-off notebook.** Use this when:
- BODACC current-year archives are already downloaded under `source-archives/bodacc/current/2026/` AND already parsed into `raw/bodacc/*.parquet` on Drive (i.e. `download_bodacc.py --mode current` ran successfully at some point).
- `clean/legal_events` on Drive is stale (max event_date < today).
- You want a single `prediction_year=2026` features partition built with cutoff = today, to score recent SIRENs.

**Self-contained** — does NOT depend on any state from `01_data_pipeline.ipynb`. Safe to delete after the refresh succeeds.

**What it does:**
1. Mirror only what's needed from Drive → local SSD: `raw/bodacc/`, `clean/`, `raw/financials/`. Skips `raw/insee/` and `raw/inpi/` because the existing `clean/` covers them.
2. Rebuild `clean/legal_events` from local raw (fast — 4 min on local SSD vs hours on Drive).
3. Build features with `cutoff_date = today` and `features_only = True` → single 2026 partition with fully-observed 12m windows.
4. Sync the new `clean/legal_events/` and `features/` partitions back to Drive.
5. Validate by inspecting FHM (`SIREN 444560502`, known liquidation).

**Expected wall time:** ~3 hours total. ~2.5h is the `raw/bodacc/` rsync (21k tiny files on Drive — bottleneck is per-file API latency, not bandwidth). Builds are 5-30 min each on local SSD.

**Don't lose the session.** Keep the tab active. Free Colab kills idle runtimes after ~90 min.

## Setup

Mounts Drive, pulls repo on `ml-workflow`, installs requirements, defines paths.

In [ ]:
from pathlib import Path
import os, subprocess, sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-workflow'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
DATA_LAKE = DRIVE_ROOT / 'data-lake'
WORK_DIR = Path('/content/pfe_work')
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')
LOCAL_DATA_LAKE = WORK_DIR / 'data-lake'
LOCAL_RAW_BODACC = LOCAL_DATA_LAKE / 'raw' / 'bodacc'
LOCAL_CLEAN = LOCAL_DATA_LAKE / 'clean'
LOCAL_RAW_FIN = LOCAL_DATA_LAKE / 'raw' / 'financials'

for p in (DATA_LAKE, WORK_DIR, DUCKDB_TMP, LOCAL_DATA_LAKE):
    p.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND_DIR / 'collabs' / 'requirements-colab.txt')])

print(f'BACKEND_DIR     = {BACKEND_DIR}')
print(f'DATA_LAKE       = {DATA_LAKE}')
print(f'LOCAL_DATA_LAKE = {LOCAL_DATA_LAKE}')
print(f'BRANCH          = {BRANCH}')

## 1 — Selective mirror Drive → local SSD

Only the directories the rest of this notebook actually reads. `raw/insee/` and `raw/inpi/` are skipped because `clean/company_identity/`, `clean/formalities_events/`, and `clean/annual_accounts/` already cover them.

**This is the slow step** (~2.5-3 hours, dominated by `raw/bodacc/` which has ~21k small files on Drive). Watch the progress bar — if it keeps advancing, it's working.

`rsync -a` is incremental — if the session dies mid-way, just re-run this cell.

In [ ]:
import shlex

print('=== Mirror raw/bodacc (slowest — ~21k small files, ~2-3 hours) ===')
LOCAL_RAW_BODACC.parent.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 "{DATA_LAKE}/raw/bodacc/" "{LOCAL_RAW_BODACC}/"

print('\n=== Mirror clean/ (fast — fewer, larger files) ===')
LOCAL_CLEAN.parent.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 "{DATA_LAKE}/clean/" "{LOCAL_CLEAN}/"

print('\n=== Mirror raw/financials ===')
if (DATA_LAKE / 'raw' / 'financials').exists():
    LOCAL_RAW_FIN.parent.mkdir(parents=True, exist_ok=True)
    !rsync -a --info=progress2 "{DATA_LAKE}/raw/financials/" "{LOCAL_RAW_FIN}/"
else:
    print('  raw/financials not on Drive — skipping. Features financial fields will be NaN.')

print('\nMirror complete. Subsequent rsyncs in future sessions are seconds (incremental).')

## 2 — Rebuild `clean/legal_events` from local raw

Uses `overwrite=True` to wipe and regenerate the legal_events clean dataset. The other clean datasets (`company_identity`, `formalities_events`, `annual_accounts`) short-circuit on the no-raw check before `_prepare_output_dir` runs — their existing local files stay untouched.

Expected: ~4-5 min reading 5500 raw bodacc parquet files on local SSD.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s', force=True)

from app.tools.build_clean_core_sources import build_clean_core_sources

summary = build_clean_core_sources(
    data_lake_dir=LOCAL_DATA_LAKE,
    overwrite=True,
    max_rows=None,
)
print('\nSummary:')
for name, info in summary.items():
    rows = info.get('rows', 'n/a')
    skipped = info.get('skipped_reason')
    if skipped:
        print(f'  {name}: SKIPPED ({skipped})')
    else:
        print(f'  {name}: rows={rows:,}')

### Sanity-check the rebuild

Confirms `clean/legal_events` now has events through today, and FHM (known liquidation in Q1 2026) is present.

In [ ]:
import duckdb

glob = (LOCAL_DATA_LAKE / 'clean' / 'legal_events' / '**' / '*.parquet').as_posix()
con = duckdb.connect()
try:
    r = con.execute(f"SELECT max(event_date), count(*) FROM read_parquet('{glob}', union_by_name=true)").fetchone()
    print(f'clean/legal_events: max event_date = {r[0]}, total rows = {r[1]:,}')
    n = con.execute(f"SELECT count(*) FROM read_parquet('{glob}', union_by_name=true) WHERE siren = '444560502'").fetchone()[0]
    print(f'\nFHM (444560502) rows in clean/legal_events: {n}')
    if n > 0:
        fhm = con.execute(f"""
            SELECT event_date, event_type, is_radiation, flag_liquidation, flag_procedure_collective
            FROM read_parquet('{glob}', union_by_name=true)
            WHERE siren = '444560502'
            ORDER BY event_date DESC
        """).df()
        from IPython.display import display
        display(fhm)
finally:
    con.close()

## 3 — Build features with mid-year cutoff

`cutoff_date = date.today()` and `features_only = True`. Builds a single `prediction_year=2026` partition where rolling-window features (`legal_risk_events_count_12m`, `days_since_last_legal_event`, etc.) end at today's date — fully observed, no partial-year bias.

Existing year-end partitions (`prediction_year=2017..2024`) on Drive are **not touched**. The new 2026 partition gets added alongside them.

Skipping labels because the 12m forward window is in the future. Do not train on this partition.

Expected: 10-30 min on local SSD.

In [ ]:
from datetime import date
from app.tools.build_company_year_features import build_company_year_datasets

build_company_year_datasets(
    data_lake_dir=LOCAL_DATA_LAKE,
    start_year=2017,           # ignored when cutoff_date is set
    end_year=2025,             # ignored when cutoff_date is set
    max_companies=None,
    year_batch_size=None,
    overwrite=False,
    cutoff_date=date.today(),
    features_only=True,
)
print(f'\nBuild done. New partition: features/company_year_features/prediction_year={date.today().year}/')

### Sanity-check FHM's feature row

Should show `prediction_year = 2026`, `legal_risk_events_count_12m >= 1`, `flag_liquidation = True` (rolled up via the BODACC join), and a small `days_since_last_legal_event` (~100 days since Jan 2026 liquidation).

In [ ]:
import duckdb

glob = (LOCAL_DATA_LAKE / 'features' / 'company_year_features' / '**' / '*.parquet').as_posix()
con = duckdb.connect()
try:
    df = con.execute(f"""
        SELECT prediction_year, prediction_date,
               administrative_status_at_cutoff, company_age_years,
               legal_events_count_all, legal_events_count_12m,
               legal_risk_events_count_12m, days_since_last_legal_event,
               annual_accounts_count_24m, latest_revenue, latest_net_result
        FROM read_parquet('{glob}', union_by_name=true)
        WHERE siren = '444560502'
        ORDER BY prediction_year DESC
    """).df()
finally:
    con.close()

from IPython.display import display
print(f'FHM feature rows: {len(df)}')
display(df)

## 4 — Sync results back to Drive

Pushes ONLY the two things that changed: `clean/legal_events/` (rebuilt with 2026) and `features/` (new 2026 partition added). The other `clean/*` datasets on Drive are not touched — even though their local manifests got rewritten as 'skipped', we don't sync them back.

`rsync` without `--delete` so existing Drive content (2017-2024 features partitions, other clean datasets) is preserved.

In [ ]:
import shlex

for relpath in ('clean/legal_events', 'features'):
    src = str(LOCAL_DATA_LAKE / relpath).rstrip('/') + '/'
    dst = str(DATA_LAKE / relpath).rstrip('/') + '/'
    (DATA_LAKE / relpath).mkdir(parents=True, exist_ok=True)
    print(f'Syncing {relpath} → Drive')
    cmd = f'rsync -a --info=progress2 {shlex.quote(src)} {shlex.quote(dst)}'
    print(f'$ {cmd}')
    !{cmd}
    print()

print('Sync complete. Drive is now current.')

## Done

Open `03_model_interrogation.ipynb`, run all cells through the helpers, then:

```python
diagnose_siren('444560502')   # BODACC section should now list the Jan 2026 liquidation_judiciaire
score_siren('444560502')       # should return prediction_year=2026, risk_band='high'
```

**Once verified, this notebook has served its purpose.** Delete it from `collabs/v1/` — future refreshes should use `01_data_pipeline.ipynb` with the right config.

**For future routine refreshes** in `01_data_pipeline.ipynb`:
- Set `USE_LOCAL_STAGING = True` so the notebook does the full Drive ↔ local mirror automatically.
- Set `DO_DOWNLOAD_BODACC = True` and the other downloads `False` if you only need to refresh BODACC.
- Set `MID_YEAR_CUTOFF_DATE = 'today'` for as-of-today scoring features.